# VLM Medical VQA Evaluation — v2 (Corrected Pipeline)

## What changed from v1 (`02_inference_harness.ipynb`)

| Change | v1 | v2 |
|---|---|---|
| Prompt format | `question + 'Answer concisely'` | MedGemma paper prompt with `Final Answer: X` |
| Answer extraction | Raw model output | Extract after `Final Answer:` |
| VQA-RAD split | `flaviagiammarino/vqa-rad` original split | Same split; contamination noted as limitation |
| Metrics | token-F1, norm-F1, BLEU, BERTScore | Same, applied to cleaner predictions |

**Why kept v1**: Documents the methodology evolution and justifies the switch.
**Reference**: MedGemma Technical Report (Google, 2026), Table 9 and Appendix Table A7.

## Cell 1 — Imports and device

In [1]:
import os, json, re, string, time
import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from PIL import Image
from datasets import load_dataset
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sacrebleu.metrics import BLEU
from tqdm import tqdm

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

Device: mps
PyTorch: 2.12.0


## Cell 2 — Image utilities

In [2]:
SLAKE_IMGS_DIR = os.path.expanduser('~/vlm_benchmark/data/slake_imgs/imgs')

def to_rgb(img: Image.Image) -> Image.Image:
    return img if img.mode == 'RGB' else img.convert('RGB')

def load_slake_image(img_name: str) -> Image.Image:
    return Image.open(os.path.join(SLAKE_IMGS_DIR, img_name)).convert('RGB')

print('Image utilities defined.')

Image utilities defined.


## Cell 3 — Corrected prompt (MedGemma paper protocol)

Source: MedGemma Technical Report, Appendix Table A7.

- **SLAKE**: `<IMAGE> + <QUESTION>` + concise answer instruction + `Final Answer: X` format
- **VQA-RAD**: `<IMAGE>` + radiology context + very short answer instruction
- Closed-ended: prepend `Answer the question with yes or no.`

In [3]:
def build_prompt_slake(question: str, is_closed: bool) -> str:
    """
    MedGemma paper prompt for SLAKE (Table A7):
    <IMAGE> + <QUESTION> + concise answer with Final Answer: X format.
    Closed-ended questions prepend yes/no instruction.
    """
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}{question} "
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"X in the format 'Final Answer: X'"
    )

def build_prompt_vqarad(question: str, is_closed: bool) -> str:
    """
    MedGemma paper prompt for VQA-RAD (Table A7):
    Given this radiology image provide a very short, definitive answer.
    """
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}Given this radiology image, which can be a frontal chest X-ray, "
        f"a single slice head or abdominal CT or MR image, provide a very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"to the following question: {question}"
    )

def extract_final_answer(text: str) -> str:
    """
    Extract content after 'Final Answer:' if present.
    Falls back to first sentence if pattern not found.
    """
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', text, re.DOTALL)
    if match:
        answer = match.group(1).strip()
        answer = re.sub(r'[\*"\']+'  , '', answer).strip()
        # Take only first line in case model wrote multiple lines after Final Answer
        answer = answer.split('\n')[0].strip()
        return answer
    # Fallback: return first sentence
    return re.split(r'(?<=[.!?])\s', text)[0].strip()

# Quick test
test_outputs = [
    'The image shows lung tissue. Final Answer: CT',
    'Based on the scan, Final Answer: **Yes**',
    'Computed tomography (CT)',   # no Final Answer — fallback
    'Final Answer: Lung\nSome extra text',
]
for t in test_outputs:
    print(f'Input:    {t[:60]}')
    print(f'Extracted: {extract_final_answer(t)}')
    print()

Input:    The image shows lung tissue. Final Answer: CT
Extracted: CT

Input:    Based on the scan, Final Answer: **Yes**
Extracted: Yes

Input:    Computed tomography (CT)
Extracted: Computed tomography (CT)

Input:    Final Answer: Lung
Some extra text
Extracted: Lung



## Cell 4 — Load VQA-RAD

Using the original `flaviagiammarino/vqa-rad` test split (451 samples).

**Note on split contamination**: MedGemma uses a decontaminated split from Yang et al. (2024)
that removes test images appearing in the training set. That split is not publicly available.
We use the original split and note this limitation when comparing VQA-RAD numbers to the
MedGemma paper. SLAKE comparisons are unaffected as it uses the standard default split.

In [4]:
vqarad      = load_dataset('flaviagiammarino/vqa-rad')
vqarad_test = vqarad['test']

yes_no  = [s for s in vqarad_test if s['answer'].strip().lower() in ('yes', 'no')]
open_q  = [s for s in vqarad_test if s['answer'].strip().lower() not in ('yes', 'no')]

print(f'VQA-RAD test: {len(vqarad_test)} samples')
print(f'  Closed (yes/no): {len(yes_no)}')
print(f'  Open-ended:      {len(open_q)}')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-eb8844602202be(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e5bc3d208bb4dee(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

VQA-RAD test: 451 samples
  Closed (yes/no): 251
  Open-ended:      200


## Cell 5 — Load SLAKE

In [5]:
slake = load_dataset('BoKelvin/SLAKE')
slake_en_test = slake['test'].filter(lambda x: x['q_lang'] == 'en')
print(f'SLAKE EN test: {len(slake_en_test)}')

open_q  = [s for s in slake_en_test if s['answer_type'] == 'OPEN']
close_q = [s for s in slake_en_test if s['answer_type'] == 'CLOSED']
print(f'  Open: {len(open_q)}, Closed: {len(close_q)}')

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

validation.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9835 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2099 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2094 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2094 [00:00<?, ? examples/s]

SLAKE EN test: 1061
  Open: 645, Closed: 416


In [6]:
# ── VQAv2 General Dataset (STREAMING) ──────────────────────────────────
# We stream the dataset to prevent downloading 20GB+ to the local disk.
# We collect exactly 500 closed + 500 open-ended samples.

from datasets import load_dataset
import random

print('Streaming VQAv2 validation split (bypassing disk storage)...')
# Added streaming=True
vqav2_stream = load_dataset('lmms-lab/VQAv2', split='validation', streaming=True)

def get_vqav2_answer(sample):
    """Extract majority vote answer from VQAv2 answer list."""
    answers = [a['answer'].strip().lower() for a in sample['answers']]
    return max(set(answers), key=answers.count)

def is_vqav2_closed(sample):
    ans = get_vqav2_answer(sample)
    return ans in ('yes', 'no')

closed_samples = []
open_samples = []

print("Extracting 1,000 balanced samples from the stream...")
# Sip from the stream until our quotas are met
for sample in vqav2_stream:
    if len(closed_samples) >= 500 and len(open_samples) >= 500:
        break # We have what we need, stop the stream!
        
    if is_vqav2_closed(sample):
        if len(closed_samples) < 500:
            closed_samples.append(sample)
    else:
        if len(open_samples) < 500:
            open_samples.append(sample)

vqav2_test = closed_samples + open_samples

# Shuffle the combined list
random.seed(42)
random.shuffle(vqav2_test)

print(f'\nVQAv2 sampled test: {len(vqav2_test)} samples')
print(f'  Closed (yes/no): {len(closed_samples)}')
print(f'  Open-ended:      {len(open_samples)}')
print(f'Sample closed: Q={vqav2_test[0]["question"]} | A={get_vqav2_answer(vqav2_test[0])}')

Streaming VQAv2 validation split (bypassing disk storage)...


README.md:   0%|          | 0.00/962 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Extracting 1,000 balanced samples from the stream...

VQAv2 sampled test: 1000 samples
  Closed (yes/no): 500
  Open-ended:      500
Sample closed: Q=How many upside-down umbrellas are visible? | A=16


In [7]:
# ── OK-VQA General Dataset (STREAMING) ─────────────────────────────────────────────
# OK-VQA requires external knowledge to answer (harder than VQAv2).
# More comparable in difficulty to medical VQA.
# We use the full test split (5,046 samples) — all open-ended,
# capped at 1,000 for consistency with other datasets.
# HF dataset: Multimodal-Fatima/OK-VQA_test
# Note: answers are a list of 10 strings (human annotators).
# Following VQA eval protocol: majority vote from the list.

from datasets import load_dataset

print('Streaming OK-VQA dataset...')
# Changed split='validation' to split='val2014'
okvqa_stream = load_dataset('lmms-lab/OK-VQA', split='val2014', streaming=True)

okvqa_test = []
print("Extracting 1,000 samples from the stream...")

for sample in okvqa_stream:
    if len(okvqa_test) >= 1000:
        break
    okvqa_test.append(sample)

print(f'OK-VQA sampled test: {len(okvqa_test)} samples')

Streaming OK-VQA dataset...


README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

Extracting 1,000 samples from the stream...
OK-VQA sampled test: 1000 samples


In [8]:
# ── General dataset prompt builders ───────────────────────────────────
# VQAv2 and OK-VQA use natural images, not medical images.
# We use the same Final Answer: X format as SLAKE for consistency,
# but remove the radiology-specific context from the VQA-RAD prompt.
# Closed-ended questions still use the yes/no prefix.

def build_prompt_vqav2(question: str, is_closed: bool) -> str:
    """
    Prompt for VQAv2 — natural image general VQA.
    Same Final Answer: X format as SLAKE for fair comparison.
    """
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}{question} "
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"X in the format 'Final Answer: X'"
    )

def build_prompt_okvqa(question: str, is_closed: bool) -> str:
    """
    Prompt for OK-VQA — knowledge-based general VQA.
    All questions are open-ended; is_closed is always False here.
    Same Final Answer: X format for consistency.
    """
    return (
        f"{question} "
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"X in the format 'Final Answer: X'"
    )

# Verify on a sample
print('VQAv2 closed prompt:')
print(build_prompt_vqav2('Is there a cat in the image?', is_closed=True))
print()
print('OK-VQA open prompt:')
print(build_prompt_okvqa('What sport is being played?', is_closed=False))

VQAv2 closed prompt:
Answer the question with yes or no. Is there a cat in the image? You may write out your argument before stating your final very short, definitive, and concise answer (if possible, a single word) X in the format 'Final Answer: X'

OK-VQA open prompt:
What sport is being played? You may write out your argument before stating your final very short, definitive, and concise answer (if possible, a single word) X in the format 'Final Answer: X'


## Cell 6 — Scoring functions (same as v1 + normalized F1)

In [9]:
stemmer   = PorterStemmer()
STOPWORDS = {'the','a','an','is','are','was','were','this','that','of','in','for'}
bleu_metric = BLEU(effective_order=True)

def tokenize_answer(text: str) -> list:
    text = re.sub(r'\*+', '', text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def normalize_answer(text: str) -> str:
    text = re.sub(r'\*+', '', text)
    text = re.split(r'(?<=[.!?])\s', text)[0]
    text = text.replace('/', ' ').lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in STOPWORDS]
    return ' '.join(tokens)

def token_f1(prediction: str, ground_truth: str) -> dict:
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def token_f1_normalized(prediction: str, ground_truth: str) -> dict:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens   = normalize_answer(ground_truth).split()
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def is_correct(prediction: str, ground_truth: str, is_closed: bool) -> bool:
    scores    = token_f1(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores['recall'] >= threshold

def compute_bleu(prediction: str, ground_truth: str) -> float:
    return bleu_metric.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    ).score

def score_jsonl(path: str) -> dict:
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if 'error' not in r]
    closed  = [r for r in records if r['is_closed']]
    open_   = [r for r in records if not r['is_closed']]

    def avg_f1(recs, fn):
        if not recs: return 0.0
        return sum(fn(r['prediction'], r['ground_truth'])['f1']
                   for r in recs) / len(recs)
    def avg_recall(recs):
        if not recs: return 0.0
        return sum(token_f1(r['prediction'], r['ground_truth'])['recall']
                   for r in recs) / len(recs)
    def accuracy(recs, is_closed, normalized=False):
        if not recs: return 0.0
        threshold = 0.5 if is_closed else 0.75
        fn = token_f1_normalized if normalized else token_f1
        return sum(
            1 for r in recs
            if fn(r['prediction'], r['ground_truth'])['recall'] >= threshold
        ) / len(recs)
    def avg_bleu(recs):
        if not recs: return 0.0
        return sum(compute_bleu(r['prediction'], r['ground_truth'])
                   for r in recs) / len(recs)

    return {
        'dataset':         os.path.basename(path),
        'n_total':         len(records),
        'n_closed':        len(closed),
        'n_open':          len(open_),
        'overall_f1':      round(avg_f1(records, token_f1) * 100, 2),
        'overall_recall':  round(avg_recall(records) * 100, 2),
        'closed_acc':      round(accuracy(closed, True)  * 100, 2),
        'open_acc':        round(accuracy(open_,  False) * 100, 2),
        'bleu':            round(avg_bleu(records), 2),
        'overall_f1_norm': round(avg_f1(records, token_f1_normalized) * 100, 2),
        'closed_acc_norm': round(accuracy(closed, True,  normalized=True) * 100, 2),
        'open_acc_norm':   round(accuracy(open_,  False, normalized=True) * 100, 2),
    }

print('Scoring functions defined.')

Scoring functions defined.


## Cell 7 — Checkpoint-aware runner (same as v1)

In [10]:
def run_dataset_with_checkpoint(
    model, processor, device, samples,
    get_image_fn, get_question_fn,
    get_answer_fn, get_is_closed_fn,
    build_prompt_fn,
    dataset_name, model_name,
    output_dir, max_samples=None
):
    os.makedirs(output_dir, exist_ok=True)
    safe_model = model_name.replace('/', '_')
    out_path   = os.path.join(output_dir, f'{safe_model}__{dataset_name}_v2.jsonl')

    if max_samples:
        samples = samples.select(range(max_samples)) if hasattr(samples, 'select') else samples[:max_samples]

    completed = {}
    if os.path.exists(out_path):
        with open(out_path, 'r') as f:
            for line in f:
                r = json.loads(line)
                completed[r['idx']] = r
        print(f'Resuming: {len(completed)} samples already done.')

    errors = 0
    f_out  = open(out_path, 'a')

    for i, sample in enumerate(tqdm(samples, desc=f'{model_name} | {dataset_name}')):
        if i in completed:
            continue
        try:
            image     = get_image_fn(sample)
            question  = get_question_fn(sample)
            answer    = get_answer_fn(sample)
            is_closed = get_is_closed_fn(sample)

            prompt_text = build_prompt_fn(question, is_closed)
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': image},
                    {'type': 'text',  'text': prompt_text},
                ]
            }]
            text   = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = processor(
                text=text, images=image, return_tensors='pt'
            ).to(device)
            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs, max_new_tokens=100, do_sample=False
                )
            input_len = inputs['input_ids'].shape[-1]
            raw       = processor.decode(
                output_ids[0][input_len:], skip_special_tokens=True
            ).strip()
            prediction = extract_final_answer(raw)

            record = {
                'idx': i, 'question': question,
                'ground_truth': answer, 'prediction': prediction,
                'raw_output': raw,
                'is_closed': is_closed,
                'model': model_name, 'dataset': dataset_name,
            }
        except Exception as e:
            errors += 1
            record = {
                'idx': i, 'question': '', 'ground_truth': '',
                'prediction': '', 'raw_output': '',
                'is_closed': False,
                'model': model_name, 'dataset': dataset_name,
                'error': str(e),
            }

        f_out.write(json.dumps(record) + '\n')
        f_out.flush()

    f_out.close()
    print(f'Done. {len(completed) + len(samples) - len(completed)} results ({errors} errors) -> {out_path}')
    return out_path

print('Runner defined.')

Runner defined.


## Cell 8 — Score existing v2 outputs (run after downloading from Drive)

v2 output files are named `*__*_v2.jsonl` to distinguish from v1.

In [18]:
output_dir = os.path.expanduser('~/vlm_benchmark/outputs')

v2_files = sorted(
    f for f in os.listdir(output_dir)
    if f.endswith('_v2.jsonl')
)

if not v2_files:
    print('No v2 output files found yet. Run inference on Colab first.')
else:
    print(f'Found {len(v2_files)} v2 output files:\n')
    for fname in v2_files:
        path   = os.path.join(output_dir, fname)
        scores = score_jsonl(path)
        print(scores)
        print()

Found 13 v2 output files:

{'dataset': 'FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__slake_7b_v2.jsonl', 'n_total': 1061, 'n_closed': 416, 'n_open': 645, 'overall_f1': 47.86, 'overall_recall': 47.68, 'closed_acc': 72.36, 'open_acc': 28.68, 'bleu': 46.71, 'overall_f1_norm': 51.08, 'closed_acc_norm': 72.36, 'open_acc_norm': 33.8}

{'dataset': 'FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__vqa_rad_7b_v2.jsonl', 'n_total': 451, 'n_closed': 251, 'n_open': 200, 'overall_f1': 57.4, 'overall_recall': 56.97, 'closed_acc': 77.69, 'open_acc': 25.5, 'bleu': 56.06, 'overall_f1_norm': 57.55, 'closed_acc_norm': 77.69, 'open_acc_norm': 26.0}

{'dataset': 'google_gemma-3-4b-it__okvqa_v2.jsonl', 'n_total': 1000, 'n_closed': 0, 'n_open': 1000, 'overall_f1': 23.95, 'overall_recall': 26.55, 'closed_acc': 0.0, 'open_acc': 24.3, 'bleu': 22.53, 'overall_f1_norm': 26.54, 'closed_acc_norm': 0.0, 'open_acc_norm': 27.1}

{'dataset': 'google_gemma-3-4b-it__slake_v2.jsonl', 'n_total': 1061, 'n_closed': 4

## Cell 9 — Compare v1 vs v2 results

Run this after you have both v1 and v2 outputs to document the improvement.

In [16]:
import pandas as pd

output_dir = os.path.expanduser('~/vlm_benchmark/outputs')

# Exclude reextracted files — they are identical to v2 and clutter the table
all_files = sorted(
    f for f in os.listdir(output_dir)
    if f.endswith('.jsonl') and 'reextracted' not in f
)

rows = []
for fname in all_files:
    path   = os.path.join(output_dir, fname)
    scores = score_jsonl(path)
    version = 'v2' if '_v2' in fname else 'v1'
    base    = fname.replace('_v2.jsonl', '').replace('.jsonl', '')
    parts   = base.split('__')
    scores['model']   = parts[0].replace('google_', 'google/')
    scores['dataset'] = parts[1] if len(parts) > 1 else 'unknown'
    scores['version'] = version
    rows.append(scores)

df = pd.DataFrame(rows)
cols = ['model', 'dataset', 'version', 'n_total',
        'overall_f1', 'overall_f1_norm', 'closed_acc', 'open_acc', 'bleu']

# Sort by dataset then model for clean reading
df = df.sort_values(['dataset', 'version', 'model']).reset_index(drop=True)
print(df[cols].to_string(index=False))

                                            model           dataset version  n_total  overall_f1  overall_f1_norm  closed_acc  open_acc  bleu
                             google/gemma-3-4b-it             okvqa      v2     1000       23.95            26.54        0.00     24.30 22.53
                llava-hf_llava-v1.6-mistral-7b-hf             okvqa      v2     1000       41.59            45.04        0.00     41.40 39.58
                             google/gemma-3-4b-it okvqa_v2_rescored      v2     1000       23.95            26.54        0.00     24.30 22.53
                llava-hf_llava-v1.6-mistral-7b-hf okvqa_v2_rescored      v2     1000       41.59            45.04        0.00     41.40 39.58
                             google/gemma-3-4b-it             slake      v1     1061       17.62            39.79       64.66     24.65  7.65
                            google/medgemma-4b-it             slake      v1     1061       55.95            57.95       76.68     47.29 50.53
      

In [13]:
# used this cell only for scoring vqav2 and okvqa for 2 general VLMs
import json, os, re, string
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sacrebleu.metrics import BLEU

# ── Number normalisation map ───────────────────────────────────────────
NUM_WORDS = {
    'zero':'0','one':'1','two':'2','three':'3','four':'4','five':'5',
    'six':'6','seven':'7','eight':'8','nine':'9','ten':'10',
    'eleven':'11','twelve':'12','thirteen':'13','fourteen':'14',
    'fifteen':'15','sixteen':'16','seventeen':'17','eighteen':'18',
    'nineteen':'19','twenty':'20'
}

def normalize_numbers(text: str) -> str:
    tokens = text.lower().split()
    return ' '.join(NUM_WORDS.get(t, t) for t in tokens)

# ── Fixed tokenizer (adds number normalisation) ────────────────────────
def tokenize_answer_fixed(text: str) -> list:
    text = re.sub(r'\*+', '', text).lower()
    text = normalize_numbers(text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1_fixed(prediction: str, ground_truth: str) -> dict:
    pred_tokens = tokenize_answer_fixed(prediction)
    gt_tokens   = tokenize_answer_fixed(ground_truth)
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def is_correct_fixed(prediction: str, ground_truth: str, is_closed: bool) -> bool:
    scores    = token_f1_fixed(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores['recall'] >= threshold

# ── Fixed extract_final_answer (same as v2 corrected version) ─────────
def extract_final_answer_fixed(text: str) -> str:
    clean = re.sub(r'\*+', '', text).strip()
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', clean, re.DOTALL)
    if match:
        answer = match.group(1).strip()
        answer = answer.split('\n')[0].strip()
        answer = answer.strip(string.punctuation + ' ')
        return answer
    return re.split(r'(?<=[.!?])\s', clean)[0].strip()

bleu_metric_fixed = BLEU(effective_order=True)

def compute_bleu_fixed(prediction: str, ground_truth: str) -> float:
    return bleu_metric_fixed.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    ).score

# ── Re-score function for general datasets ────────────────────────────
def rescore_general_jsonl(path: str) -> dict:
    """
    Re-extracts predictions from raw_output and re-scores with
    fixed tokenizer. Writes corrected records to *_rescored.jsonl.
    """
    records  = [json.loads(l) for l in open(path)]
    records  = [r for r in records if 'error' not in r]

    out_path = path.replace('.jsonl', '_rescored.jsonl')
    fixed    = 0

    with open(out_path, 'w') as f_out:
        for r in records:
            raw      = r.get('raw_output', '')
            new_pred = extract_final_answer_fixed(raw) if raw else r['prediction']

            if new_pred != r['prediction']:
                fixed += 1

            corrected = dict(r)
            corrected['original_prediction'] = r['prediction']
            corrected['prediction']          = new_pred
            f_out.write(json.dumps(corrected) + '\n')

    # Score the rescored file
    rescored = [json.loads(l) for l in open(out_path)]
    closed   = [r for r in rescored if r['is_closed']]
    open_    = [r for r in rescored if not r['is_closed']]

    def avg_f1(recs):
        if not recs: return 0.0
        return sum(token_f1_fixed(r['prediction'], r['ground_truth'])['f1']
                   for r in recs) / len(recs)

    def accuracy(recs, is_closed):
        if not recs: return 0.0
        return sum(is_correct_fixed(r['prediction'], r['ground_truth'], is_closed)
                   for r in recs) / len(recs)

    def avg_bleu(recs):
        if not recs: return 0.0
        return sum(compute_bleu_fixed(r['prediction'], r['ground_truth'])
                   for r in recs) / len(recs)

    print(f'\n{os.path.basename(path)}')
    print(f'  Predictions re-extracted: {fixed}/{len(records)}')
    return {
        'dataset':      os.path.basename(out_path),
        'n_total':      len(rescored),
        'n_closed':     len(closed),
        'n_open':       len(open_),
        'overall_f1':   round(avg_f1(rescored) * 100, 2),
        'closed_acc':   round(accuracy(closed, True)  * 100, 2),
        'open_acc':     round(accuracy(open_,  False) * 100, 2),
        'bleu':         round(avg_bleu(rescored), 2),
    }

# ── Run on all general dataset JSONL files ────────────────────────────
OUTPUT_DIR = os.path.expanduser('~/vlm_benchmark/outputs')

general_files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith('.jsonl')
    and ('vqav2' in f or 'okvqa' in f)
    and 'rescored' not in f
)

print(f'Found {len(general_files)} general dataset files:\n')
results = []
for fname in general_files:
    path   = os.path.join(OUTPUT_DIR, fname)
    scores = rescore_general_jsonl(path)
    print(scores)
    results.append(scores)

Found 4 general dataset files:


google_gemma-3-4b-it__okvqa_v2.jsonl
  Predictions re-extracted: 0/1000
{'dataset': 'google_gemma-3-4b-it__okvqa_v2_rescored.jsonl', 'n_total': 1000, 'n_closed': 0, 'n_open': 1000, 'overall_f1': 23.95, 'closed_acc': 0.0, 'open_acc': 24.3, 'bleu': 22.53}

google_gemma-3-4b-it__vqav2_v2.jsonl
  Predictions re-extracted: 0/1000
{'dataset': 'google_gemma-3-4b-it__vqav2_v2_rescored.jsonl', 'n_total': 1000, 'n_closed': 500, 'n_open': 500, 'overall_f1': 58.81, 'closed_acc': 76.8, 'open_acc': 41.0, 'bleu': 54.4}

llava-hf_llava-v1.6-mistral-7b-hf__okvqa_v2.jsonl
  Predictions re-extracted: 0/1000
{'dataset': 'llava-hf_llava-v1.6-mistral-7b-hf__okvqa_v2_rescored.jsonl', 'n_total': 1000, 'n_closed': 0, 'n_open': 1000, 'overall_f1': 41.64, 'closed_acc': 0.0, 'open_acc': 41.5, 'bleu': 39.58}

llava-hf_llava-v1.6-mistral-7b-hf__vqav2_v2.jsonl
  Predictions re-extracted: 0/1000
{'dataset': 'llava-hf_llava-v1.6-mistral-7b-hf__vqav2_v2_rescored.jsonl', 'n_total': 1000,

In [14]:
!pip install -q bert-score

In [15]:
import os
import json
import pandas as pd
import torch
from bert_score import score

output_dir = os.path.expanduser('~/vlm_benchmark/outputs')

# 1. Grab all relevant .jsonl files
all_files = sorted(
    f for f in os.listdir(output_dir)
    if f.endswith('.jsonl') and 'reextracted' not in f
)

print(f"Found {len(all_files)} files. Calculating BERTScore (this may take a moment)...\n")

rows = []
for fname in all_files:
    path = os.path.join(output_dir, fname)
    
    # 2. Extract metadata from filename
    version = 'v2' if '_v2' in fname else 'v1'
    base = fname.replace('_v2.jsonl', '').replace('.jsonl', '')
    parts = base.split('__')
    model_name = parts[0].replace('google_', 'google/')
    dataset_name = parts[1] if len(parts) > 1 else 'unknown'
    
    # 3. Read the JSONL records
    with open(path, 'r') as f:
        records = [json.loads(line) for line in f]
        
    # 4. Filter out any records that had errors
    records = [r for r in records if 'error' not in r]
    if not records:
        continue
        
    # 5. Extract predictions and ground truths as lists of strings
    predictions = [str(r['prediction']) for r in records]
    ground_truths = [str(r['ground_truth']) for r in records]
    
    # 6. Compute BERTScore 
    # lang='en' defaults to RoBERTa-large. If it runs out of memory on MPS/T4, 
    # you can add model_type='distilbert-base-uncased' as an argument for speed.
    P, R, F1 = score(predictions, ground_truths, lang='en', verbose=False)
    
    # Calculate the mean F1 score and convert to percentage
    mean_bert_f1 = F1.mean().item()
    
    rows.append({
        'model': model_name,
        'dataset': dataset_name,
        'version': version,
        'n_total': len(records),
        'bertscore_f1': round(mean_bert_f1 * 100, 2)
    })
    
    print(f"✅ Computed {model_name} | {dataset_name} | {version}")

# 7. Build and display the final DataFrame
df_bert = pd.DataFrame(rows)
df_bert = df_bert.sort_values(['dataset', 'version', 'model']).reset_index(drop=True)

print("\n--- BERTScore Results ---")
print(df_bert.to_string(index=False))

Found 22 files. Calculating BERTScore (this may take a moment)...



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

✅ Computed llava-hf_llava-v1.6-mistral-7b-hf | okvqa | v2


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f8c98064-4d4b-4e46-b706-9247f3c25978)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /roberta-large/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: f8f87a69-263f-4abd-b71e-952e49f5db47)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probabl

✅ Computed llava-hf_llava-v1.6-mistral-7b-hf | okvqa_v2_rescored | v2


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 6f134f42-4344-4809-8454-c0b5ef000dd2)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /roberta-large/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: dbc2c3c5-3345-4a5f-ae4e-92ab199fd5b6)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /roberta-large/resolve/main/tokenizer_config.json (Caused by NameResolutionErro

KeyboardInterrupt: 